In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os, random, pandas as pd, numpy as np
from collections import defaultdict

random.seed(42)
np.random.seed(42)
SEED = 42
PATCHES_PER_PATIENT = 2000

# ── Paths ─────────────────────────────────────────────────
a2_blocks = "/kaggle/input/datasets/anishapanja/bc-xai-a2-2000-partial/A2_full/BLOCKS_NORM_MACENKO"
e2_blocks  = "/kaggle/input/datasets/anishapanja/bc-xai-e2-2000/tiles/E2/extracted_full/BLOCKS_NORM_MACENKO"
a2_clini_path = "/kaggle/input/datasets/anishapanja/tcga-brca-a2-clini/TCGA-BRCA-A2-CLINI.xlsx"
e2_clini_path = "/kaggle/input/datasets/anishapanja/tcga-brca-e2-clini/TCGA-BRCA-E2-CLINI.xlsx"

# ── Build patch records ───────────────────────────────────
def build_patch_records(blocks_path, institution, max_patches=PATCHES_PER_PATIENT):
    records = []
    for slide_folder in os.listdir(blocks_path):
        patient_id = slide_folder[:12]
        slide_path = os.path.join(blocks_path, slide_folder)
        all_patches = [
            os.path.join(slide_path, f)
            for f in os.listdir(slide_path)
            if f.endswith(".jpg")
        ]
        selected = (random.sample(all_patches, max_patches)
                    if len(all_patches) > max_patches
                    else all_patches)
        for p in selected:
            records.append({"patient_id": patient_id,
                            "institution": institution,
                            "patch_path": p})
    return pd.DataFrame(records)

print("Building A2 patches...")
a2_df = build_patch_records(a2_blocks, "A2")
print(f"A2: {len(a2_df)} patches, {a2_df['patient_id'].nunique()} patients")

print("Building E2 patches...")
e2_df = build_patch_records(e2_blocks, "E2")
print(f"E2: {len(e2_df)} patches, {e2_df['patient_id'].nunique()} patients")

all_patches = pd.concat([a2_df, e2_df], ignore_index=True)

# ── Load CLINI labels ─────────────────────────────────────
def clean_clini(path):
    df = pd.read_excel(path)
    df["subtype_clean"] = df["TCGA Subtype"].str.replace("BRCA.", "", regex=False)
    df = df[df["subtype_clean"] != "Normal"].copy()
    return df[["PATIENT", "subtype_clean", "TIL Regional Fraction"]]

clini = pd.concat([clean_clini(a2_clini_path),
                   clean_clini(e2_clini_path)], ignore_index=True)
print(f"\nCLINI patients: {len(clini)}")
print(clini["subtype_clean"].value_counts())

# ── Merge labels ──────────────────────────────────────────
final = all_patches.merge(clini, left_on="patient_id",
                          right_on="PATIENT", how="inner")
final = final[["patient_id", "institution", "subtype_clean",
               "TIL Regional Fraction", "patch_path"]].reset_index(drop=True)
print(f"\nFinal labeled patches: {len(final)}")
print(f"Unique patients: {final['patient_id'].nunique()}")

# ── Stratified patient-level split ───────────────────────
rng = np.random.RandomState(SEED)
patient_subtype = final.drop_duplicates("patient_id")[["patient_id","subtype_clean"]]
train_p, val_p, test_p = [], [], []

for subtype in sorted(patient_subtype["subtype_clean"].unique()):
    pts = sorted(patient_subtype[patient_subtype["subtype_clean"]==subtype]["patient_id"].tolist())
    shuffled = rng.permutation(pts)
    n = len(shuffled)
    n_test = max(1, int(round(n * 0.15)))
    n_val  = max(1, int(round(n * 0.15)))
    test_p.extend(shuffled[:n_test])
    val_p.extend(shuffled[n_test:n_test+n_val])
    train_p.extend(shuffled[n_test+n_val:])

test_set  = set(test_p)
val_set   = set(val_p)

def assign_split(pid):
    if pid in test_set: return "test"
    if pid in val_set:  return "val"
    return "train"

final["split"] = final["patient_id"].apply(assign_split)

print("\nSplit distribution (patients):")
print(final.groupby(["split","subtype_clean"])["patient_id"].nunique())
print("\nSplit distribution (patches):")
print(final.groupby("split")["patch_path"].count())

# ── Save ──────────────────────────────────────────────────
os.makedirs("/kaggle/working", exist_ok=True)
out_path = "/kaggle/working/patches_with_split_2000.csv"
final.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")
print(f"Size: {os.path.getsize(out_path)/(1024*1024):.1f} MB")
print("\nDone.")